# Unitary Patent procedural step date: `REG723_PROC_STEP_DATE`
The `REG723_PROC_STEP_DATE` table contains key information about the dates of procedural steps, complementing the `REG721_PROC_STEP table`. Specifically, it records the dates associated with each procedural step, providing essential temporal data, such as when specific actions in the patent granting process occurred. Each record links a procedural step to its corresponding date, helping to track the timeline of the patent application process. Let’s break down the key fields in this table. The first step as usual is to initialise the PATSTAT client and import the table.

In [14]:
from epo.tipdata.patstat import PatstatClient

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

# Importing the models
from epo.tipdata.patstat.database.models import REG722_PROC_STEP_TEXT,REG721_PROC_STEP, REG723_PROC_STEP_DATE

# Importing sql alchemy package
from sqlalchemy import func

## Key fields in the `REG723_PROC_STEP_DATE` table
* `ID` (primary key): a technical identifier for the Unitary Patent, without business meaning.
* `STEP_ID`: this attribute contains IDs identifying a procedural step.
* `STEP_DATE`: this attribute stores the date associated with a procedural step in the patent process. It records the exact date on which a specific procedural step took place, providing essential temporal information for tracking the step's timeline.
* `STEP_DATE_TYPE`: this attribute specifies the type of date related to a procedural step. It indicates the event or role associated with the date, offering a better understanding of the timing and context of the step. The attribute can take the following values.
  
    * `CANCT`: date of cancellation
    * `DISPA`: date of dispatch
    * `DRAEX`: decision date of request for accelerated examination (RAEX)
    * `EFFEC`: date effective
    * `GRNTF`: grant fee paid
    * `LAPPR`: date of later approval
    * `PAYM1`: date of payment 1
    * `PAYMN`: date of payment
    * `PRNTF`: print fee paid
    * `RAEXA`: date of request for accelerated examination (RAEX)
    * `RECPT`: date of receipt
    * `REPLY`: date of reply
    * `REQST`: date of request
    * `RESLT`: result date
    * `TCLMS`: translation of claims
    * `WV713`: waiver agreement, according to rule 71(3)
    * `DTREG`: date of registration
    * `DTDEC`: date of decision
    * `DTNOT`: date of notification
    * `LCNSC`: date of licence commitment
    * `DTREC`: date of receipt
    * `WDRWL`: date of withdrawal
    * `FIACT`: date of filing action
    * `FIDEC`: date of filing decision
    * `FIAPP`: date of filing appeal


Let's plot the table to visualize its structure.

In [15]:
reg723 = (
    db.query(
        REG723_PROC_STEP_DATE.id,
        REG723_PROC_STEP_DATE.step_id,
        REG723_PROC_STEP_DATE.step_date,
        REG723_PROC_STEP_DATE.step_date_type,
    )
    .order_by(REG723_PROC_STEP_DATE.id)
    .limit(10000)
)

reg723_df = patstat.df(reg723)
reg723_df

,id,step_id,step_date,step_date_type
0,5018904,STEP_UP_UDFI_5,2024-06-13,REQST
1,5018904,STEP_UP_UDEC_6,2024-08-14,DTDEC
2,5701869,STEP_UP_UDWI_2,2023-10-02,DISPA
3,5701869,STEP_UP_UDFI_1,2023-07-14,REQST
4,6748170,STEP_UP_UDFI_9,2023-09-30,REQST
...,...,...,...,...
9995,15178522,STEP_UP_UDEC_11636,2024-04-22,DTDEC
9996,15178638,STEP_UP_UDEC_11688,2024-11-11,DTDEC
9997,15178638,STEP_UP_UREG_11689,2024-11-11,DTREG
9998,15178638,STEP_UP_RFEE_11690,2025-05-27,PAYMN


## Focus on `STEP_DATE_TYPE`
Let's plot the distinct types of step date in the `REG723_PROC_STEP_DATE` table.

In [5]:
reg723_dt = db.query(
        REG723_PROC_STEP_DATE.step_date_type
    ).distinct()
reg723_dt_df = patstat.df(reg723_dt)
reg723_dt_df


,step_date_type
0,PAYMN
1,DTNOT
2,EFFEC
3,REQST
4,DTREG
5,WDRWL
6,DTDEC
7,DISPA


Let’s join the `REG723_PROC_STEP_DATE` table with the `REG721_PROC_STEP` table to obtain a complete overview of the information related to each procedural step.

In [12]:
reg723 = (
    db.query(
         REG723_PROC_STEP_DATE.id,
         #REG723_PROC_STEP_DATE.step_id,
         REG723_PROC_STEP_DATE.step_date,
         REG723_PROC_STEP_DATE.step_date_type,
         REG721_PROC_STEP.step_id,
         REG721_PROC_STEP.step_phase,
         REG721_PROC_STEP.step_code,
         REG721_PROC_STEP.step_result,
         REG721_PROC_STEP.step_result_type,
            
    ).join(
         REG723_PROC_STEP_DATE,
          REG723_PROC_STEP_DATE.id== REG721_PROC_STEP.id
              
    ).order_by(REG723_PROC_STEP_DATE.id)
    .limit(10000)
)

reg723_df = patstat.df(reg723)
reg723_df
        

,id,step_date,step_date_type,step_id,step_phase,step_code,step_result,step_result_type
0,5018904,2024-06-13,REQST,STEP_UP_UDEC_6,UNIP,UDEC,Negative,
1,5018904,2024-06-13,REQST,STEP_UP_UDFI_5,UNIP,UDFI,,
2,5018904,2024-08-14,DTDEC,STEP_UP_UDFI_5,UNIP,UDFI,,
3,5018904,2024-08-14,DTDEC,STEP_UP_UDEC_6,UNIP,UDEC,Negative,
4,5701869,2023-10-02,DISPA,STEP_UP_UDFI_1,UNIP,UDFI,,
...,...,...,...,...,...,...,...,...
9995,12720125,2025-03-17,PAYMN,STEP_UP_UREG_3184,UNIP,UREG,,
9996,12720125,2024-04-26,PAYMN,STEP_UP_RFEE_3185,UNIP,RFEE,,
9997,12720125,2023-06-07,EFFEC,STEP_UP_RFEE_3185,UNIP,RFEE,,
9998,12720125,2023-06-27,REQST,STEP_UP_RFEE_3185,UNIP,RFEE,,
